# MNS2026008 — Colab re-run
Run the cells in order. Upload `AITR_MNS2026008_Research_Package_v4_05Nov2026.zip` when asked. Total time: about 5 minutes.

In [ ]:
from google.colab import files
up = files.upload()  # choose the v4 research package ZIP
zipname = list(up)[0]
!unzip -q -o "$zipname" -d /content/pkg
%cd /content/pkg/AITR_MNS2026008_Working_v4

## 1. Install and test (Prompt 08 / 11 evidence)

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pytest -q tests 2>&1 | tail -3
!python -m py_compile src/*.py && echo COMPILE_OK

In [ ]:
!python -c "import platform,sklearn,pandas,numpy;print(platform.python_version(),sklearn.__version__,pandas.__version__,numpy.__version__)" | tee colab_environment.txt
!pip freeze > colab_pip_freeze.txt
!echo saved

## 2. Re-run the analysis on the current labels
Uses the final reviewed labels (keep the `ai` flag: 2,305 off-topic rows remain AI first-pass labels).

In [ ]:
IN='data/collection_audit_20261031/data/private/model_input_reviewed_20261105_private.csv'
!bash scripts/run_all.sh $IN colab_rerun ai 2>&1 | tail -5
!cat results/colab_rerun_crossfit/crossfit_paired_contrasts.csv

## 3. (After you fill the review-first sheet) merge your decisions and re-run
Upload your completed `AITR_Review_First_Sheet_20261105.xlsx`.

In [ ]:
up2 = files.upload()
sheet = list(up2)[0]
OUT='data/collection_audit_20261031/data/private/model_input_reviewed.csv'
!python src/merge_review_first.py --sheet "$sheet" --ai-labels data/ai_provisional_20261105/ai_provisional_screening_labels.csv --queue data/collection_audit_20261031/data/private/candidate_pool_human_screening_queue_private.csv --out $OUT
!bash scripts/run_all.sh $OUT reviewed ai 2>&1 | tail -5
!cat results/reviewed_crossfit/crossfit_paired_contrasts.csv

## 4. Download results

In [ ]:
!zip -qr colab_outputs.zip results colab_environment.txt colab_pip_freeze.txt data/collection_audit_20261031/data/private/*.summary.json 2>/dev/null
files.download('colab_outputs.zip')